# Prepare MIO-TCD for Ultralytics YOLO
Converts mapped GT into YOLO labels while retaining raw images outside the repository. Input: frozen manifest, GT and raw `train/`; output: labels and `mio_tcd.yaml`. Image lists reference resolved raw paths, so no bulk image copy is made.

## Config

In [ ]:
DATASET_ROOT_OVERRIDE = None
FORCE_REBUILD = False

## Imports, validation and processing

In [ ]:
from pathlib import Path
import sys, time, pandas as pd
sys.path.insert(0, str(Path.cwd()))
from mio_tcd_utils import PROJECT_ROOT, resolve_dataset_root, read_annotations, prepare_yolo, TARGET_NAMES
DATASET_ROOT = resolve_dataset_root(DATASET_ROOT_OVERRIDE)
manifest_path = PROJECT_ROOT / 'data/mio_tcd/splits/split_manifest.csv'
if not manifest_path.is_file(): raise FileNotFoundError('Run 02_mio_split.ipynb first.')
manifest = pd.read_csv(manifest_path, dtype={'image_id': str})
annotations = read_annotations(DATASET_ROOT)
started = time.perf_counter()
yolo_root = prepare_yolo(annotations, manifest, FORCE_REBUILD)
print(f'Prepare + single validation completed in {(time.perf_counter() - started) / 60:.1f} minutes')
print('Class order:', dict(enumerate(TARGET_NAMES))); print('Saved:', yolo_root / 'mio_tcd.yaml')

## Summary

In [ ]:
for split in ('train', 'val', 'test'):
    print(split, 'images=', int((manifest.split == split).sum()), 'labels=', len(list((yolo_root/'labels'/split).glob('*.txt'))))
print((yolo_root/'mio_tcd.yaml').read_text())